In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

In [2]:
from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------------
# Stage 7 output directory
# ------------------------------------------------------------------

stage7_dir = (
        DATA_ROOT
        / "processed"
        / "stage_9_track_stitching"
)

# ------------------------------------------------------------------
# Load detections
# ------------------------------------------------------------------

detections = pd.read_csv(
    stage7_dir / "detections.csv"
)

# ------------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------------

tracks = pd.read_csv(
    stage7_dir / "tracks.csv"
)

# ------------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------------

with open(stage7_dir / "metadata.json") as f:
    metadata = json.load(f)

print(f"Loaded {len(detections):,} detections")
print(f"Loaded {len(tracks):,} track records")
print(metadata)

Loaded 4,223 detections
Loaded 4,223 track records
{'motion_compensation': True, 'global_motion_estimator': 'median (iteratively refined on matched pairs)', 'gap_closing': True, 'max_gap': 2, 'stitch_max_distance': 15.0, 'merge_detection': True, 'merge_max_distance': 15.0, 'merge_volume_tolerance': 0.25}


In [3]:
# ============================================================
# Build track summary
# ============================================================

# Add volume information to each tracked point
track_points = (
    tracks.merge(
        detections[
            ["frame", "cell_id", "volume_voxels"]
        ],
        left_on=["frame", "cell"],
        right_on=["frame", "cell_id"],
        how="left",
    )
)

track_summary = (
    track_points
    .sort_values(["track_id", "frame"])
    .groupby("track_id")
    .agg(
        # Lifetime
        start_frame=("frame", "first"),
        end_frame=("frame", "last"),
        length=("frame", "count"),

        # Detection IDs
        start_cell=("cell", "first"),
        end_cell=("cell", "last"),

        # Start position
        start_z=("z", "first"),
        start_y=("y", "first"),
        start_x=("x", "first"),

        # End position
        end_z=("z", "last"),
        end_y=("y", "last"),
        end_x=("x", "last"),

        # Volume
        start_volume=("volume_voxels", "first"),
        end_volume=("volume_voxels", "last"),
    )
    .reset_index()
)

track_summary.head()

,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
0,0,0,6,7,0,2,0.231183,6.354839,52.204301,0.607375,38.255965,63.403471,440.0,625.0
1,1,0,19,20,1,66,0.153153,6.815315,71.774775,16.943574,60.206897,110.907524,186.0,1083.0
2,2,0,19,19,2,40,0.320833,26.012500,65.191667,8.797089,64.744863,86.595034,222.0,835.0
3,3,0,19,20,3,80,1.074627,62.743555,58.230665,21.535495,113.673094,107.260298,240.0,868.0
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0


In [4]:
# ============================================================
# Build frame indices
# ============================================================

tracks_starting = {}
tracks_ending = {}

for frame, group in track_summary.groupby("start_frame"):
    tracks_starting[frame] = group.copy()

for frame, group in track_summary.groupby("end_frame"):
    tracks_ending[frame] = group.copy()

print(f"{len(tracks_starting)} start-frame groups")
print(f"{len(tracks_ending)} end-frame groups")

20 start-frame groups
20 end-frame groups


In [5]:
# ============================================================
# Candidate parents
# ============================================================

last_frame = detections["frame"].max()

candidate_parents = track_summary[
    track_summary["end_frame"] < last_frame
    ].copy()

print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

Candidate parents: 174


,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
0,0,0,6,7,0,2,0.231183,6.354839,52.204301,0.607375,38.255965,63.403471,440.0,625.0
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0
8,8,0,18,19,8,49,0.363905,186.692308,38.485207,13.902646,226.693122,69.422222,166.0,1114.0
10,10,0,5,6,10,24,0.497959,246.916327,67.051020,5.179648,250.879397,74.844221,128.0,644.0
12,12,0,0,1,12,12,1.788909,35.497317,70.622540,1.788909,35.497317,70.622540,518.0,518.0


In [6]:
BOUNDARY_MARGIN_Z = 3
BOUNDARY_MARGIN_Y = 10
BOUNDARY_MARGIN_X = 10

In [7]:
def touches_boundary(row):
    return (
            row["end_z"] <= BOUNDARY_MARGIN_Z
            or row["end_z"] >= 63 - BOUNDARY_MARGIN_Z
            or row["end_y"] <= BOUNDARY_MARGIN_Y
            or row["end_y"] >= 255 - BOUNDARY_MARGIN_Y
            or row["end_x"] <= BOUNDARY_MARGIN_X
            or row["end_x"] >= 255 - BOUNDARY_MARGIN_X
    )

In [8]:
candidate_parents = candidate_parents[
    ~candidate_parents.apply(touches_boundary, axis=1)
]

In [9]:
print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

Candidate parents: 43


,track_id,start_frame,end_frame,length,start_cell,end_cell,start_z,start_y,start_x,end_z,end_y,end_x,start_volume,end_volume
4,4,0,18,19,4,27,0.364431,98.571429,34.836735,7.040041,160.604723,47.205339,737.0,636.0
8,8,0,18,19,8,49,0.363905,186.692308,38.485207,13.902646,226.693122,69.422222,166.0,1114.0
13,13,0,15,16,13,67,1.993144,93.462292,58.957884,18.701071,135.928919,63.779942,559.0,688.0
28,28,0,4,5,28,46,9.363100,79.296125,65.798231,13.481534,95.527699,67.099432,877.0,1244.0
32,32,0,5,6,32,47,7.178119,206.491863,86.828210,13.214106,219.633501,92.727960,1183.0,717.0


In [10]:
print(f"Total tracks: {len(track_summary)}")

track_summary["length"].describe()

Total tracks: 398


count    398.000000
mean      10.610553
std        7.334511
min        1.000000
25%        4.000000
50%        9.500000
75%       19.000000
max       20.000000
Name: length, dtype: float64